# E6 — reavaliação em 177.490 (uniformização, tarefa 20260822-1915)

Re-avalia as 42 curvas do E6 (10 células de `tab:e6` + 32 curvas com semente,
entropia e aleatório × 2 classificadores × 8 sementes) no denominador único
**177.490**, no lugar dos 181.490 atuais. **CPU só — sklearn, sem GPU.**

**Método (roteiro do `revisor1`, tarefa 1900): re-avaliação, nunca re-seleção.**
A seleção de cada curva está CONGELADA em `*_state.json` (`labeled_idx`, os
50.000 índices do pool na ordem escolhida pelo seletor original). Para cada
checkpoint |L| que já existe na curva publicada, este notebook retreina o
classificador com o MESMO prefixo (`labeled_idx[:k]`) e reavalia só as
métricas externas — nos dois denominadores (177.490 e 181.490 inteiro) no
mesmo passe, persistindo as predições por instância do checkpoint final.
`acc_int`/`f1_int` são transportados sem recálculo (o pool não muda).

Resultados **ao lado** dos antigos (sufixo `_pop177490`), nada sobrescrito.
Retomada automática por checkpoint: se a sessão cair, a próxima rodada pula
tudo que já está no `.jsonl` de saída.


In [ ]:
# 1) Configuração
BRANCH = "claude/e3prime-seed-7-bx08ks"   # o runner reescreve esta linha


In [ ]:
# 2) Clonar o repositório (vai para /tmp, fora da saída do kernel)
import os, subprocess
REPO = "https://github.com/GHDaru/activelearning.git"
REPO_DIR = "/tmp/activelearning"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())


In [ ]:
# 3) Dependências — o sklearn já vem na imagem padrão do Kaggle; confere
import importlib
for mod in ("sklearn", "numpy"):
    importlib.import_module(mod)
    print(mod, "ok")


In [ ]:
# 4) A execução — as 42 curvas, retomada automática por checkpoint.
#
#    --out-dir aponta DIRETO pra /kaggle/working: é o único diretório que
#    sobrevive ao fim da sessão e que `kaggle kernels output` consegue baixar.
#    Escrever primeiro no clone (/tmp) e só copiar no final foi o bug da v1
#    deste notebook — se a sessão do Kaggle corta o processo antes de ele
#    terminar as 42 curvas (bem provável, a campanha inteira passa de 20h),
#    a cópia final nunca roda e TODO o progresso da sessão se perde. Escrever
#    direto em /kaggle/working faz cada checkpoint sobreviver assim que é
#    gravado, igual ao padrão já usado no e3prime_kaggle.ipynb (célula 5).
#
#    subprocess (não !python) de propósito: lista de argumentos explícita e
#    código de saída real, como o padrão do E3'/E1E4.
import subprocess, sys, time

OUT = "/kaggle/working/e6_results"
t0 = time.time()
cmd = [sys.executable, "experiments/e6population/reavaliar_177490.py",
       "--all-tab-e6", "--all-seeded", "--out-dir", OUT]
print("comando:", " ".join(cmd))
proc = subprocess.run(cmd)
print(f"\nsaiu com código {proc.returncode} em {(time.time() - t0)/3600:.2f} h")


In [ ]:
# 5) Conferência final — o output já está em /kaggle/working/e6_results
#    (célula 4 escreveu direto lá); esta célula só lista o que existe.
import glob, os

OUT = "/kaggle/working/e6_results"
novos = sorted(glob.glob(f"{OUT}/*_pop177490*.jsonl"))
print(f"{len(novos)} arquivo(s) em {OUT}:")
for p in novos:
    print(" ", os.path.basename(p), os.path.getsize(p), "bytes")
